# Baseline Runs By Beta

Run the non-RL baselines on the validation weather data in `dataset/val.csv` for beta values `0.01`, `0.025`, `0.03`, and `0.05`. Each run is saved as a pickle in `notebook_results/` with the pattern `(model)_(beta).pkl`.


In [ ]:
import pickle
from pathlib import Path
import os

import pandas as pd
import numpy as np

from ascab.env.env import ActionConstrainer, MultipleWeatherASCabEnv, get_dates
from ascab.train import CeresOptimizer, NaiveUmbrellaAgent, OneAgent, RandomAgent, UmbrellaAgent, ZeroAgent
from ascab.utils.weather import WeatherDataLibrary

from ascab.env.env import (
    MultipleWeatherASCabEnv,
    get_default_end_of_season,
    get_default_start_of_season,
    get_weather_library,
    get_weather_library_from_csv
)


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset" / "val.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "notebook_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CERES_DIR = PROJECT_ROOT / "ascab" / "ceres"
CERES_COUNTRY = "ES1"

# VAL_CSV_PATH = PROJECT_ROOT / "dataset" / "val.csv"
# VAL_LOCATION_ID = "42.162_3.0924"
ES_LOCATION = (42.1620, 3.0924)
FR_LOCATION = (44.0986, 1.1628)
ES1_LOCATION = (37.8880, -4.7790)

BETA_VALUES = [
    # 0.01,
    0.025,
    # 0.03,
    # 0.05
]
TEST_YEARS = [year for year in range(2016, 2025) if year % 2 != 0]
RANDOM_RUNS = 10
RANDOM_SEED_STREAM = 107

BASELINES = {
    "Farmer's Practice": lambda env: NaiveUmbrellaAgent(ActionConstrainer(env=env), render=False),
    "Super Farmer": lambda env: UmbrellaAgent(ActionConstrainer(env=env), render=False),
    "Zero": lambda env: ZeroAgent(ActionConstrainer(env=env), render=False),
}

VAL_CSV_PATH


In [ ]:
def load_weather_library_from_csv(
    csv_path: Path,
    *,
    location_id: str | None = None,
    years: list[int] | None = None,
    location_col: str = "location",
    index_col: str = "date",
) -> WeatherDataLibrary:
    df = pd.read_csv(csv_path, parse_dates=[index_col])
    if location_id is not None:
        df = df[df[location_col] == location_id].copy()
    if df.empty:
        raise ValueError(f"No rows found in {csv_path} for location_id={location_id!r}")

    df[index_col] = pd.to_datetime(df[index_col], utc=True, errors="coerce")
    df = df.dropna(subset=[index_col]).set_index(index_col).sort_index()
    df["Year"] = df.index.year
    if years is not None:
        df = df[df["Year"].isin(years)].copy()
    if df.empty:
        raise ValueError(f"No rows found in {csv_path} for years={years}")

    weather_library = WeatherDataLibrary()
    for (loc_id, year), year_df in df.groupby([location_col, "Year"], sort=True):
        weather_df = year_df.drop(columns=[location_col, "Year"], errors="ignore")
        params = {
            "latitude": None,
            "longitude": None,
            "start_date": weather_df.index.min().strftime("%Y-%m-%d"),
            "end_date": weather_df.index.max().strftime("%Y-%m-%d"),
        }
        key = f"{loc_id}_{year}_{params['start_date']}_{params['end_date']}"
        weather_library.collect_weather(params, key=key, loaded_weather=weather_df)

    return weather_library


def make_es_test_env(beta: float) -> MultipleWeatherASCabEnv:
    return MultipleWeatherASCabEnv(
        # weather_data_library=get_weather_library_from_csv(
        #     csv_path=os.path.join(os.chdir(), "dataset", "val.csv")
        # ),
        weather_data_library=get_weather_library(
            locations=[ES1_LOCATION],
            dates=get_dates(
                TEST_YEARS,
                start_of_season=get_default_start_of_season(),
                end_of_season=get_default_end_of_season(),
            ),
        ),
        biofix_date="March 10",
        budbreak_date="March 10",
        mode="sequential",
        discrete_actions=False,
        beta=beta,
    )


def beta_label(beta: float) -> str:
    return f"{beta}"


def result_path(model_name: str, beta: float) -> Path:
    return OUTPUT_DIR / f"{model_name}_{beta_label(beta)}.pkl"


def make_es_year_env(year: int, beta: float) -> MultipleWeatherASCabEnv:
    return MultipleWeatherASCabEnv(
        weather_data_library=get_weather_library(
            locations=[ES1_LOCATION],
            dates=get_dates(
                [year],
                start_of_season=get_default_start_of_season(),
                end_of_season=get_default_end_of_season(),
            ),
        ),
        biofix_date="March 10",
        budbreak_date="March 10",
        mode="sequential",
        discrete_actions=False,
        beta=beta,
    )


def ceres_beta_label(beta: float) -> str:
    return f"{beta}".replace(".", "p")


def ceres_solution_path(year: int, beta: float, country: str = CERES_COUNTRY) -> Path:
    return CERES_DIR / f"ceres_{year}_{country}_beta{ceres_beta_label(beta)}.txt"


def describe_ceres_mask(label: str, indices: np.ndarray, one_agent_results: pd.DataFrame) -> str:
    if len(indices) == 0 or "Date" not in one_agent_results:
        return f"{label}: count={len(indices)}"
    dates = pd.to_datetime(one_agent_results.iloc[indices]["Date"])
    return f"{label}: count={len(indices)}, first={dates.min().date()}, last={dates.max().date()}"


def expand_ceres_indices_at_boundaries(
    indices: np.ndarray,
    action_length: int,
    target_length: int,
) -> np.ndarray | None:
    if len(indices) == 0 or target_length < len(indices):
        return None

    missing = target_length - len(indices)
    if missing == 0:
        return indices

    before_available = indices[0]
    after_available = action_length - indices[-1] - 1
    before_count = min((missing + 1) // 2, before_available)
    after_count = min(missing - before_count, after_available)

    remaining = missing - before_count - after_count
    if remaining > 0:
        add_before = min(remaining, before_available - before_count)
        before_count += add_before
        remaining -= add_before
    if remaining > 0:
        add_after = min(remaining, after_available - after_count)
        after_count += add_after
        remaining -= add_after
    if remaining > 0:
        return None

    before = np.arange(indices[0] - before_count, indices[0]) if before_count else np.array([], dtype=int)
    after = np.arange(indices[-1] + 1, indices[-1] + after_count + 1) if after_count else np.array([], dtype=int)
    return np.r_[before, indices, after]


def get_ceres_unmasked_indices(
    one_agent_results: pd.DataFrame,
    loaded_actions: int,
    solution_path: Path,
) -> np.ndarray:
    candidates = [
        ("Action > 0", one_agent_results["Action"].to_numpy() > 0),
    ]
    if "InfectionWindow" in one_agent_results:
        candidates.append(("InfectionWindow == 1", one_agent_results["InfectionWindow"].to_numpy() == 1))

    counts = {}
    candidate_indices = {}
    for label, mask in candidates:
        indices = np.flatnonzero(mask)
        counts[label] = len(indices)
        candidate_indices[label] = indices
        if len(indices) == loaded_actions:
            if label != "Action > 0":
                print(
                    f"Using {label} for {solution_path.name}: "
                    f"loaded_actions={loaded_actions}, Action > 0 count={counts['Action > 0']}"
                )
            return indices

    max_boundary_expansion = 3
    for label, indices in candidate_indices.items():
        missing_actions = loaded_actions - len(indices)
        if 0 < missing_actions <= max_boundary_expansion:
            expanded = expand_ceres_indices_at_boundaries(
                indices,
                action_length=len(one_agent_results),
                target_length=loaded_actions,
            )
            if expanded is not None and len(expanded) == loaded_actions:
                print(
                    f"Warning: {solution_path.name} has {missing_actions} more actions than the current {label} mask. "
                    f"Using a boundary-expanded contiguous window. "
                    f"{describe_ceres_mask(label, indices, one_agent_results)}; "
                    f"expanded_count={len(expanded)}"
                )
                return expanded

    mask_descriptions = [
        describe_ceres_mask(label, indices, one_agent_results)
        for label, indices in candidate_indices.items()
    ]
    raise ValueError(
        f"{solution_path.name} has {loaded_actions} actions, but none of the current masks match. "
        f"Mask counts: {counts}. Details: {mask_descriptions}. This usually means the txt file "
        f"was generated with a different action constraint or weather/year setup."
    )


def load_ceres_year_result(year: int, beta: float, country: str = CERES_COUNTRY) -> pd.DataFrame:
    solution_path = ceres_solution_path(year, beta, country=country)
    if not solution_path.exists():
        raise FileNotFoundError(f"Missing Ceres solution file: {solution_path}")

    env = ActionConstrainer(make_es_year_env(year, beta))
    optimizer = CeresOptimizer(env, str(solution_path))

    one_agent_results = OneAgent(ascab=env, render=False).run()
    optimizer.optimized_actions = np.atleast_1d(np.loadtxt(solution_path))
    optimizer.unmasked_indices = get_ceres_unmasked_indices(
        one_agent_results,
        loaded_actions=len(optimizer.optimized_actions),
        solution_path=solution_path,
    )
    optimizer.action_length = len(one_agent_results)

    year_result = optimizer.run_ceres_agent()
    year_result["Model"] = "Ceres"
    year_result["Beta"] = beta
    year_result["Year"] = year
    year_result["CeresFile"] = solution_path.name
    return year_result


def run_ceres_from_txt(beta: float, country: str = CERES_COUNTRY) -> pd.DataFrame:
    year_results = [load_ceres_year_result(year, beta, country=country) for year in TEST_YEARS]
    return pd.concat(year_results, ignore_index=True)


In [ ]:
all_results = {}

for beta in BETA_VALUES:
    print(f"Running Ceres at beta={beta_label(beta)} from {CERES_DIR}")
    ceres_result = run_ceres_from_txt(beta, country=CERES_COUNTRY)
    path = result_path("Ceres", beta)
    with open(path, "wb") as f:
        pickle.dump(ceres_result, f)

    all_results[("Ceres", beta)] = ceres_result
    print(f"Saved {path}")

    for model_name, make_agent in BASELINES.items():
        print(f"Running {model_name} at beta={beta_label(beta)}")
        env = make_es_test_env(beta)
        agent = make_agent(env)
        result = agent.run()
        result["Model"] = model_name
        result["Beta"] = beta

        path = result_path(model_name, beta)
        with open(path, "wb") as f:
            pickle.dump(result, f)

        all_results[(model_name, beta)] = result
        print(f"Saved {path}")

    print(f"Running Random at beta={beta_label(beta)} for {RANDOM_RUNS} seeds")
    rng = np.random.RandomState(seed=RANDOM_SEED_STREAM)
    random_results_by_run = {}

    for run_idx in range(RANDOM_RUNS):
        random_seed = int(rng.randint(0, 100))
        print(f"  Random run {run_idx}: seed={random_seed}")
        env = make_es_test_env(beta)
        random_agent = RandomAgent(
            ascab=ActionConstrainer(env=env),
            render=False,
            seed=random_seed,
        )
        random_result = random_agent.run()
        random_result["Model"] = "Random"
        random_result["Beta"] = beta
        random_result["RandomRun"] = run_idx
        random_result["RandomSeed"] = random_seed
        random_results_by_run[run_idx] = random_result

    path = result_path("Random", beta)
    with open(path, "wb") as f:
        pickle.dump(random_results_by_run, f)

    all_results[("Random", beta)] = random_results_by_run
    print(f"Saved {path}")


In [ ]:
summary_rows = []
for (model_name, beta), result in all_results.items():
    if isinstance(result, dict):
        result_items = result.items()
    else:
        result_items = [(None, result)]

    for run_idx, run_result in result_items:
        summary_rows.append(
            {
                "Model": model_name,
                "Beta": beta,
                "Run": run_idx,
                "Rows": len(run_result),
                "Years": sorted(run_result["Year"].dropna().unique().tolist()) if "Year" in run_result else None,
                "Path": str(result_path(model_name, beta)),
            }
        )

summary = pd.DataFrame(summary_rows)
summary


In [ ]:
summary_frames = []
for (model_name, beta), result in all_results.items():
    if isinstance(result, dict):
        result_items = result.items()
    else:
        result_items = [(None, result)]

    for run_idx, run_result in result_items:
        frame = run_result.copy()
        frame["Model"] = model_name
        frame["Beta"] = beta
        frame["Run"] = run_idx
        summary_frames.append(frame)

summary = pd.concat(summary_frames, ignore_index=True)

mean_summary = summary.groupby(["Model", "Beta"]).agg(
    Reward=("Reward", "median"),
    Risk=("Risk", "median"),
    Action=("Action", "median"),
    Rows=("Reward", "size"),
)
mean_summary


In [ ]:
bootstrap_frames = []
for (model_name, beta), result in all_results.items():
    if isinstance(result, dict):
        result_items = result.items()
    else:
        result_items = [(None, result)]

    for run_idx, run_result in result_items:
        frame = run_result.copy()
        frame["Date"] = pd.to_datetime(frame["Date"])
        if "Year" not in frame:
            frame["Year"] = frame["Date"].dt.year
        frame["Model"] = model_name
        frame["Beta"] = beta
        frame["Run"] = run_idx
        bootstrap_frames.append(frame)

bootstrap_daily_results = pd.concat(bootstrap_frames, ignore_index=True)

bootstrap_episode_results = (
    bootstrap_daily_results
    .groupby(["Model", "Beta", "Run", "Year"], dropna=False)
    .agg(
        CumulativeReward=("Reward", "sum"),
        PesticideUse=("Action", "sum"),
        RiskIndex=("Risk", "sum"),
        Rows=("Reward", "size"),
    )
    .reset_index()
)


def bootstrap_median_ci(values, n_boot: int = 10_000, ci: float = 0.95, seed: int = 107):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    if len(values) == 1:
        value = float(values[0])
        return value, value, value

    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(n_boot, len(values)), replace=True)
    boot_medians = np.median(samples, axis=1)
    alpha = (1.0 - ci) / 2.0
    lo, hi = np.quantile(boot_medians, [alpha, 1.0 - alpha])
    return float(np.median(values)), float(lo), float(hi)


metrics = ["CumulativeReward", "PesticideUse", "RiskIndex"]
bootstrap_rows = []
for (model_name, beta), group in bootstrap_episode_results.groupby(["Model", "Beta"], dropna=False):
    for metric in metrics:
        median, ci_low, ci_high = bootstrap_median_ci(group[metric])
        bootstrap_rows.append(
            {
                "Model": model_name,
                "Beta": beta,
                "Metric": metric,
                "Median": median,
                "CI95Low": ci_low,
                "CI95High": ci_high,
                "N": len(group),
            }
        )

bootstrap_summary = (
    pd.DataFrame(bootstrap_rows)
    .sort_values(["Beta", "Model", "Metric"])
    .reset_index(drop=True)
)

bootstrap_summary


In [ ]:
# Saved RL agent inference helpers
import re

from gymnasium.wrappers import FlattenObservation
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from sb3_contrib import RecurrentPPO

SAVED_RL_MODELS_DIR = PROJECT_ROOT / "ascab" / "additional_runs"
SAVED_RL_SEED_PATTERN = re.compile(r"seed(?P<seed>\d+)")
LOCATION_COORDS = {
    "ES": ES_LOCATION,
    "FR": FR_LOCATION,
    "ES1": ES1_LOCATION,
}


def extract_saved_rl_seed(model_path: Path) -> str:
    match = SAVED_RL_SEED_PATTERN.search(model_path.stem)
    if match is None:
        raise ValueError(f"Could not extract seed from model filename: {model_path.name}")
    return match.group("seed")


def find_saved_rl_model_paths(models_dir: Path = SAVED_RL_MODELS_DIR) -> list[Path]:
    return sorted(models_dir.glob("rl_agent_RecurrentPPO_seed*.zip"))


def saved_rl_info_to_dataframe(info: dict) -> pd.DataFrame:
    ignored_keys = {"TimeLimit.truncated", "episode", "terminal_observation"}
    result = {key: value for key, value in info.items() if key not in ignored_keys}
    return pd.DataFrame(result).assign(Date=lambda x: pd.to_datetime(x["Date"]))


def make_saved_rl_test_env(
    *,
    location_label: str,
    beta: float = 0.025,
    risk_period: bool = False,
) -> MultipleWeatherASCabEnv | ActionConstrainer:
    location_label = location_label.upper()
    if location_label not in LOCATION_COORDS:
        raise ValueError(f"Unknown location_label={location_label!r}. Use one of {sorted(LOCATION_COORDS)}")

    env = MultipleWeatherASCabEnv(
        weather_data_library=get_weather_library(
            locations=[LOCATION_COORDS[location_label]],
            dates=get_dates(
                TEST_YEARS,
                start_of_season=get_default_start_of_season(),
                end_of_season=get_default_end_of_season(),
            ),
        ),
        biofix_date="March 10",
        budbreak_date="March 10",
        mode="sequential",
        discrete_actions=True,
        beta=beta,
    )
    return ActionConstrainer(env, risk_period=risk_period) if risk_period else env


def make_saved_rl_vec_env(
    *,
    norm_path: Path,
    location_label: str,
    beta: float = 0.025,
    risk_period: bool = False,
) -> tuple[VecNormalize, int]:
    env = make_saved_rl_test_env(location_label=location_label, beta=beta, risk_period=risk_period)
    n_eval_episodes = len(env.unwrapped.weather_keys)
    env = FlattenObservation(env)
    env = Monitor(env)
    vec_env = DummyVecEnv([lambda: env])
    vec_env = VecNormalize.load(str(norm_path), vec_env)
    vec_env.training = False
    vec_env.norm_reward = False
    return vec_env, n_eval_episodes


def run_saved_rl_model(
    model_path: Path,
    norm_path: Path,
    env: VecNormalize,
    n_eval_episodes: int,
) -> pd.DataFrame:
    model = RecurrentPPO.load(str(model_path), env=env, print_system_info=False)
    episode_results = []

    for _ in range(n_eval_episodes):
        observation = env.reset()
        lstm_states = None
        episode_starts = np.ones((env.num_envs,), dtype=bool)
        done = np.array([False])
        info = None

        while not done[0]:
            action, lstm_states = model.predict(
                observation,
                state=lstm_states,
                episode_start=episode_starts,
                deterministic=True,
            )
            observation, _, done, infos = env.step(action)
            episode_starts = done
            info = infos[0]

        episode_results.append(saved_rl_info_to_dataframe(info))

    return pd.concat(episode_results, ignore_index=True)


def print_saved_rl_cumulative_reward_by_year(result: pd.DataFrame, seed: str) -> None:
    rewards_by_year = (
        result.assign(Year=lambda df: pd.to_datetime(df["Date"]).dt.year)
        .groupby("Year", sort=True)["Reward"]
        .sum()
    )
    print(f"Cumulative reward by year for saved RL seed {seed}:")
    for year, reward in rewards_by_year.items():
        print(f"  {year}: {reward:.6f}")


def run_saved_rl_agents(
    *,
    location_label: str = "ES",
    beta: float = 0.025,
    risk_period: bool = False,
    models_dir: Path = SAVED_RL_MODELS_DIR,
    output_root: Path = OUTPUT_DIR,
) -> dict[str, pd.DataFrame]:
    location_label = location_label.upper()
    output_dir = output_root / location_label
    output_dir.mkdir(parents=True, exist_ok=True)

    model_paths = find_saved_rl_model_paths(models_dir)
    if not model_paths:
        raise FileNotFoundError(f"No RecurrentPPO model .zip files found in {models_dir}")

    saved_results = {}
    for model_path in model_paths:
        seed = extract_saved_rl_seed(model_path)
        norm_path = model_path.with_name(f"{model_path.stem}_norm.pkl")
        if not norm_path.exists():
            raise FileNotFoundError(f"Missing normalization file for {model_path.name}: {norm_path}")

        print(f"Running saved RL seed {seed} at {location_label} with beta={beta}, risk_period={risk_period}")
        env, n_eval_episodes = make_saved_rl_vec_env(
            norm_path=norm_path,
            location_label=location_label,
            beta=beta,
            risk_period=risk_period,
        )
        result = run_saved_rl_model(model_path, norm_path, env, n_eval_episodes)
        result["Seed"] = int(seed)
        result["Model"] = "RL"
        result["ModelPath"] = str(model_path)
        result["NormPath"] = str(norm_path)
        result["RiskPeriodConstrained"] = risk_period
        result["LocationLabel"] = location_label
        result["Latitude"] = LOCATION_COORDS[location_label][0]
        result["Longitude"] = LOCATION_COORDS[location_label][1]
        result["Beta"] = beta

        print_saved_rl_cumulative_reward_by_year(result, seed)
        save_path = output_dir / f"rl_agent_RecurrentPPO_seed{seed}.pkl"
        with open(save_path, "wb") as f:
            pickle.dump(result, f)
        print(f"Saved {save_path}")
        saved_results[seed] = result

    return saved_results


# Example:
# saved_rl_results = run_saved_rl_agents(location_label="ES", beta=0.025, risk_period=False)


In [ ]:
es = run_saved_rl_agents(location_label="ES", beta=0.025, risk_period=False)

fr = run_saved_rl_agents(location_label="FR", beta=0.025, risk_period=False)

es_ood = run_saved_rl_agents(location_label="ES1", beta=0.025, risk_period=False)

